# 🪑💤 Detector: Persona Sentada + Dormida

**Detecta:**
1. ✅ Cuando una persona se sienta
2. ✅ Cuando se queda dormida

**Métricas de somnolencia:**
- EAR (ojos cerrados)
- PERCLOS (% tiempo ojos cerrados)
- Frecuencia parpadeo
- Inclinación cabeza
- Bostezos

**Alertas:** 🟢 Despierto → 🟡 Somnoliento → 🟠 Microsueño → 🔴 Dormido

In [ ]:
%%capture
!pip uninstall -y mediapipe opencv-python opencv-python-headless numpy protobuf
!pip install mediapipe==0.10.13 numpy==1.26.4 opencv-python==4.8.1.78 protobuf==4.25.3 matplotlib==3.9.0
print("✅ Instalado")

In [ ]:
from google.colab import files
import cv2, math, pandas as pd, matplotlib.pyplot as plt, numpy as np
import mediapipe as mp
from collections import deque
from IPython.display import Video, display

print("✅ Imports")

In [ ]:
# CONFIGURACIÓN
EAR_THRESH = 0.21
MICROSLEEP_FRAMES = 45  # 1.5s
SLEEP_FRAMES = 150      # 5s
PERCLOS_ALERT = 0.20
PERCLOS_SLEEP = 0.40
HEAD_PITCH_MAX = 25.0
MAR_THRESH = 0.6

# Postura
HIP_DROP_SIT = 0.62
KNEE_MIN, KNEE_MAX = 75, 120

print(f"✅ Config: EAR={EAR_THRESH}, Microsleep={MICROSLEEP_FRAMES}f, Sleep={SLEEP_FRAMES}f")

In [ ]:
print("📌 Sube tu video:")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"✅ Video: {video_path}")

In [ ]:
# Funciones auxiliares
def dist(a,b): return math.dist(a,b)

def ear(lm, w, h, eye):
    pts = [(lm[i].x*w, lm[i].y*h) for i in eye]
    v = dist(pts[2], pts[3])
    h_dist = dist(pts[0], pts[1]) + 1e-6
    return v/h_dist, pts

def mar(lm, w, h, mouth):
    pts = [(lm[i].x*w, lm[i].y*h) for i in mouth]
    v = dist(pts[2], pts[3])
    h_dist = dist(pts[0], pts[1]) + 1e-6
    return v/h_dist

def head_pose(lm, w, h):
    nose = np.array([lm[1].x*w, lm[1].y*h])
    chin = np.array([lm[152].x*w, lm[152].y*h])
    vec = chin - nose
    pitch = np.degrees(np.arctan2(vec[1], np.linalg.norm(vec)+1e-6))
    
    l_eye = np.array([lm[33].x*w, lm[33].y*h])
    r_eye = np.array([lm[263].x*w, lm[263].y*h])
    eye_vec = r_eye - l_eye
    roll = np.degrees(np.arctan2(eye_vec[1], eye_vec[0]+1e-6))
    return pitch, roll

def angle3p(a, b, c):
    ba = a - b
    bc = c - b
    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)
    if norm_ba < 1e-6 or norm_bc < 1e-6:
        return 0.0
    cos_ang = np.dot(ba, bc) / (norm_ba * norm_bc)
    return float(np.degrees(np.arccos(np.clip(cos_ang, -1, 1))))

print("✅ Funciones cargadas")

In [ ]:
# PROCESAMIENTO
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out_name = "video_sentado_dormido.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(out_name, fourcc, fps, (w,h))

mp_pose = mp.solutions.pose.Pose(model_complexity=1, min_detection_confidence=0.5)
mp_face = mp.solutions.face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

# Landmarks
R_EYE = [33,133,159,145]
L_EYE = [362,263,386,374]
MOUTH = [61,291,13,14]

# Estado
sit_state = "STANDING"
sit_baseline = None
sit_samples = []
ema_hip, ema_knee, ema_torso = None, None, None

frames_closed = 0
eye_states = deque(maxlen=int(60*fps))  # 60s window
blink_times = deque(maxlen=100)

# Resultados
results_data = []
frame_id = 0

print("🎬 Procesando video...")
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_id += 1
    t = frame_id / fps
    
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # === DETECCIÓN DE POSTURA ===
    pose_res = mp_pose.process(rgb)
    sitting = False
    knee_deg = 180
    
    if pose_res.pose_landmarks:
        lm_pose = pose_res.pose_landmarks.landmark
        
        # Keypoints
        r_hip = np.array([lm_pose[24].x*w, lm_pose[24].y*h])
        r_knee = np.array([lm_pose[26].x*w, lm_pose[26].y*h])
        r_ankle = np.array([lm_pose[28].x*w, lm_pose[28].y*h])
        r_shoulder = np.array([lm_pose[12].x*w, lm_pose[12].y*h])
        
        knee_deg = angle3p(r_hip, r_knee, r_ankle)
        hip_y_norm = lm_pose[24].y
        
        # EMA
        ema_hip = hip_y_norm if ema_hip is None else 0.8*ema_hip + 0.2*hip_y_norm
        ema_knee = knee_deg if ema_knee is None else 0.8*ema_knee + 0.2*knee_deg
        
        # Baseline
        if ema_knee > 150:
            sit_samples.append(ema_hip)
            if len(sit_samples) >= int(fps) and sit_baseline is None:
                sit_baseline = np.median(sit_samples)
        
        baseline = sit_baseline if sit_baseline else ema_hip
        hip_drop = ema_hip / max(baseline, 1e-6)
        
        # Estado sentado
        if hip_drop >= HIP_DROP_SIT and KNEE_MIN <= ema_knee <= KNEE_MAX:
            sit_state = "SITTING"
            sitting = True
        elif hip_drop < 0.55 or ema_knee > 130:
            sit_state = "STANDING"
        
        # Dibujar pose
        mp_draw.draw_landmarks(frame, pose_res.pose_landmarks, mp.solutions.pose.POSE_CONNECTIONS)
    
    # === DETECCIÓN DE SOMNOLENCIA (solo si sentado) ===
    sleep_state = "N/A"
    ear_val = 0
    perclos = 0
    blink_rate = 0
    alert = False
    
    if sitting:
        face_res = mp_face.process(rgb)
        
        if face_res.multi_face_landmarks:
            lm_face = face_res.multi_face_landmarks[0].landmark
            
            # EAR
            ear_l, pts_l = ear(lm_face, w, h, L_EYE)
            ear_r, pts_r = ear(lm_face, w, h, R_EYE)
            ear_val = (ear_l + ear_r) / 2
            
            # Estado ojos
            eyes_closed = ear_val < EAR_THRESH
            eye_states.append(1 if eyes_closed else 0)
            
            if eyes_closed:
                frames_closed += 1
            else:
                if frames_closed >= 10:  # Parpadeo
                    blink_times.append(t)
                frames_closed = 0
            
            # PERCLOS
            if len(eye_states) > 0:
                perclos = sum(eye_states) / len(eye_states)
            
            # Blink rate
            if len(blink_times) >= 2:
                time_window = t - blink_times[0]
                if time_window > 0:
                    blink_rate = (len(blink_times) / time_window) * 60
            
            # Head pose
            pitch, roll = head_pose(lm_face, w, h)
            
            # MAR
            mar_val = mar(lm_face, w, h, MOUTH)
            
            # Determinar estado
            if frames_closed >= SLEEP_FRAMES:
                sleep_state = "ASLEEP"
                alert = True
            elif frames_closed >= MICROSLEEP_FRAMES:
                sleep_state = "MICROSLEEP"
                alert = True
            elif perclos >= PERCLOS_SLEEP:
                sleep_state = "ASLEEP"
                alert = True
            elif perclos >= PERCLOS_ALERT or abs(pitch) > HEAD_PITCH_MAX or mar_val > MAR_THRESH:
                sleep_state = "DROWSY"
                alert = True
            else:
                sleep_state = "AWAKE"
            
            # Dibujar ojos
            for pts in [pts_l, pts_r]:
                eye_poly = np.array([[pts[0]], [pts[2]], [pts[1]], [pts[3]]], np.int32)
                cv2.polylines(frame, [eye_poly], True, (0,255,0), 2)
    
    # === VISUALIZACIÓN ===
    # Banner
    if sitting:
        if alert:
            if sleep_state == "ASLEEP":
                color = (0,0,255)  # Rojo
            elif sleep_state == "MICROSLEEP":
                color = (0,140,255)  # Naranja
            else:
                color = (0,200,200)  # Amarillo
        else:
            color = (0,180,0)  # Verde
        
        overlay = frame.copy()
        cv2.rectangle(overlay, (0,0), (w,100), color, -1)
        frame = cv2.addWeighted(overlay, 0.3, frame, 0.7, 0)
        
        cv2.putText(frame, f"SENTADO - {sleep_state}", (20,50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255,255,255), 3)
        
        if alert:
            cv2.putText(frame, "⚠️ ALERTA SOMNOLENCIA ⚠️", (20,90), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 3)
    else:
        cv2.putText(frame, "DE PIE", (20,50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (180,180,180), 3)
    
    # Métricas
    y_pos = 130
    if sitting:
        cv2.putText(frame, f"EAR: {ear_val:.3f}", (20,y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        y_pos += 30
        cv2.putText(frame, f"PERCLOS: {perclos*100:.1f}%", (20,y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        y_pos += 30
        cv2.putText(frame, f"Parpadeos/min: {blink_rate:.1f}", (20,y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    
    writer.write(frame)
    
    # Log datos
    results_data.append({
        'time_s': round(t, 2),
        'sitting': sitting,
        'sleep_state': sleep_state,
        'ear': round(ear_val, 3),
        'perclos': round(perclos, 3),
        'blink_rate': round(blink_rate, 1),
        'knee_deg': round(knee_deg, 1),
        'alert': alert
    })
    
    if frame_id % 100 == 0:
        print(f"  Frame {frame_id}/{total_frames} ({frame_id/total_frames*100:.1f}%)")

cap.release()
writer.release()
mp_pose.close()
mp_face.close()

print("\n✅ Procesamiento completado")
print(f"   Video guardado: {out_name}")

In [ ]:
# RESULTADOS
df = pd.DataFrame(results_data)

# Filtrar solo cuando está sentado
df_sitting = df[df['sitting'] == True].copy()

if len(df_sitting) > 0:
    print("\n📊 ESTADÍSTICAS:")
    print(f"  Tiempo total sentado: {len(df_sitting)/fps:.1f}s")
    
    # Contar estados
    for state in ['AWAKE', 'DROWSY', 'MICROSLEEP', 'ASLEEP']:
        count = len(df_sitting[df_sitting['sleep_state'] == state])
        if count > 0:
            pct = count / len(df_sitting) * 100
            print(f"  {state}: {count/fps:.1f}s ({pct:.1f}%)")
    
    # Alertas
    alerts = len(df_sitting[df_sitting['alert'] == True])
    print(f"  ⚠️ Alertas: {alerts/fps:.1f}s ({alerts/len(df_sitting)*100:.1f}%)")
    
    print("\n📋 Primeros 10 registros sentado:")
    print(df_sitting.head(10))
else:
    print("⚠️ No se detectó a nadie sentado en el video")

# Guardar CSV
df.to_csv('resultados_completos.csv', index=False)
print("\n✅ CSV guardado: resultados_completos.csv")

In [ ]:
# GRÁFICOS
if len(df_sitting) > 0:
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    
    # EAR
    axes[0].plot(df_sitting['time_s'], df_sitting['ear'], label='EAR', linewidth=2)
    axes[0].axhline(EAR_THRESH, color='red', linestyle='--', label='Threshold')
    axes[0].set_ylabel('EAR')
    axes[0].set_title('Eye Aspect Ratio')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # PERCLOS
    axes[1].plot(df_sitting['time_s'], df_sitting['perclos']*100, label='PERCLOS', color='orange', linewidth=2)
    axes[1].axhline(PERCLOS_ALERT*100, color='yellow', linestyle='--', label='Alert')
    axes[1].axhline(PERCLOS_SLEEP*100, color='red', linestyle='--', label='Sleep')
    axes[1].set_ylabel('PERCLOS (%)')
    axes[1].set_title('Percentage of Eye Closure')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    # Estado
    state_map = {'AWAKE': 0, 'DROWSY': 1, 'MICROSLEEP': 2, 'ASLEEP': 3, 'N/A': -1}
    df_sitting['state_num'] = df_sitting['sleep_state'].map(state_map)
    axes[2].plot(df_sitting['time_s'], df_sitting['state_num'], label='Sleep State', linewidth=2)
    axes[2].set_ylabel('Estado')
    axes[2].set_yticks([0, 1, 2, 3])
    axes[2].set_yticklabels(['Awake', 'Drowsy', 'Microsleep', 'Asleep'])
    axes[2].set_xlabel('Tiempo (s)')
    axes[2].set_title('Estado de Somnolencia')
    axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('graficos_somnolencia.png', dpi=150)
    plt.show()
    
    print("✅ Gráfico guardado: graficos_somnolencia.png")

In [ ]:
# REPRODUCIR VIDEO
print("📹 Video procesado:")
display(Video(out_name, width=720, embed=True))

# DESCARGAR
print("\n📥 Descargando archivos...")
files.download(out_name)
files.download('resultados_completos.csv')
files.download('graficos_somnolencia.png')
print("✅ Descargas completadas")